In [3]:
from dotenv import load_dotenv
load_dotenv(dotenv_path = ".env")

import os
from typing_extensions import TypedDict, Annotated
from langchain.messages import SystemMessage, AIMessage, HumanMessage, ToolMessage
from typing import Literal, Any, Sequence, List 
from langgraph.graph import StateGraph, START, END

from langchain_groq import ChatGroq

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENROUTER_API = os.getenv("OPENROUTER_API")
LANGSMITH_API = os.getenv("LANGSMITH_API")

# """
if GROQ_API_KEY and OPENROUTER_API and LANGSMITH_API:
    print("Successful import")
    # """

llm = ChatGroq(
    api_key= GROQ_API_KEY,
    model_name = "llama-3.3-70b-versatile",
    temperature=0.7
)
print(llm.invoke("Hi, tell me about yourself").content)

Successful import
I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.memory import ConversationBufferMemory

while True:
    query = input("you: ")
    history = []
    

In [4]:
llm.invoke("What's date today").content

"I'm not currently able to share the time."

In [3]:
import tools

In [1]:
import importlib
import workflow_without_llm
import tools
import json
importlib.reload(workflow_without_llm)
importlib.reload(tools)

atg_agent = workflow_without_llm.atg_agent

In [5]:
log_data = tools.collect_logs(path = r"data/monitoring_logs.txt")
collection = tools.create_or_get_cromadb(logs = log_data)

In [7]:
query = "Memory leak suspected"
result = tools.search_query(query = query, collection= collection, n_results = 3)
result

{'ids': [['14', '10', '32']],
 'embeddings': None,
 'documents': [['2026-05-26 09:10:05 - MONITORING_TOOL - ERROR - Memory leak suspected in analytics engine, heap usage grew from 2GB to 12GB within 30 minutes.',
   '2026-05-26 09:10:01 - MONITORING_TOOL - INFO - CPU utilization stable at 45% across cluster nodes, memory usage averaging 65% with no anomalies detected.',
   '2026-05-26 09:30:03 - MONITORING_TOOL - WARN - CPU spike detected on node-12, utilization jumped from 40% to 90% within 2 minutes.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None, None, None]],
 'distances': [[0.7879611253738403, 1.259913682937622, 1.4553433656692505]]}

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env")

import os, json
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

class State(TypedDict):
    history: list  
    query: str           
    answer: str        
    use_tool: bool      


GROQ_API_KEY = os.getenv("GROQ_API_KEY")
llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model_name="openai/gpt-oss-20b",
    temperature=0.7
)

class Tools:
    def __init__(self, collection):
        self.collection = collection

    def search_query(self, query: str, n_results: int = 3):
        results = self.collection.query(
            query_texts=[query],
            n_results=n_results
        )
        return results
    
tools = Tools(collection)

class Decision(BaseModel):
    use_tool: bool = Field(description="Whether to call the tool")
    answer: str = Field(description="Direct answer if applicable")

parser = JsonOutputParser(pydantic_object=Decision)

prompt = PromptTemplate(
    template="""
    Conversation history: {history}
    User query: {query}
    
    Decide: Should I answer directly or use the tool?
    Respond strictly in JSON.
    """,
    input_variables=["history", "query"],
    output_parser=parser,
)

chain = prompt | llm | parser

def chat_node(state: State) -> State:
    print("\n[CHAT NODE]")
    history_text = "\n".join([str(msg) for msg in state["history"]])
    result: Decision = chain.invoke({"history": history_text, "query": state["query"]})

    state["use_tool"] = result["use_tool"]
    state["answer"] = result["answer"]
    state["history"].append(AIMessage(content=state["answer"]))

    return state



def tool_node(state: State) -> State:
    print("\n[TOOL NODE]")
    results = tools.search_query(query=state["query"], n_results=3)
    tool_msg = ToolMessage(content=str(results), tool_call_id="search_query")
    state["history"].append(tool_msg)

    prompt = PromptTemplate(
        template="""
        User query: {query}
        Tool results: {results}

        Write a helpful natural language answer.
        """,
        input_variables=["query", "results"]
    )

    chain_tool = prompt | llm
    response = chain_tool.invoke({"query": state["query"], "results": results})
    state["answer"] = response.content
    state["history"].append(AIMessage(content=state["answer"]))
    return state



workflow = StateGraph(State)
workflow.add_node("chat_node", chat_node)
workflow.add_node("tool_node", tool_node)

workflow.add_edge(START, "chat_node")
workflow.add_conditional_edges("chat_node", lambda s: "tool_node" if s["use_tool"] else END)
workflow.add_edge("tool_node", END)

agent = workflow.compile()


def run():
    state: State = {"history": [], "query": "", "answer": "", "use_tool": False}
    while True:
        query = input("\nYou: ")
        if query.lower() in ["quit", "exit"]:
            print("Goodbye!")
            break

        # Save HumanMessage
        state["query"] = query
        state["history"].append(HumanMessage(content=query))

        result = agent.invoke(state)

        print("Assistant:", result["answer"], f"Tool used: [{result['use_tool']}]")
        state.update(result)

run()



[CHAT NODE]
Assistant: Hello Rahul! How can I assist you today? Tool used: False

[CHAT NODE]
Assistant: You are Rahul. Tool used: False

[CHAT NODE]
Assistant:  Tool used: False

[CHAT NODE]
Assistant: You asked me to write exactly what my last reply was. I responded with the text: "You are Rahul. I am an AI assistant. I do not have a memory of your identity beyond this conversation." Tool used: False

[CHAT NODE]

[TOOL NODE]
Assistant: Yes, there have been a couple of recent failures reported for the system on **2026‑05‑26**:

| Time | Tool | Severity | Issue | Impact |
|------|------|----------|-------|--------|
| 09:00:02 | **DATABASE_TOOL** | ERROR | Transaction rollback triggered due to a deadlock between the `payment_processing` and `inventory_update` services during peak load. | Short‑term data consistency issue; transactions that hit the deadlock were rolled back. |
| 09:25:03 | **MONITORING_TOOL** | ERROR | Service `payment-gateway` latency exceeded the SLA threshold of 500